# Graphagate Training & Evaluation

This notebook trains and tests the **Temporal Graph Network** (TGN) model
locally, without Docker. It uses the available accelerator: **MPS** on Apple
Silicon, **CUDA** on NVIDIA GPUs, otherwise CPU (the Device Detection cell
reports it). On Apple Silicon it applies a temporary monkey-patch of
`tgn.scatter` for a known PyTorch MPS bug with `int64` on
`scatter_reduce_`; on CUDA the patch is inert.


In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import sys

if torch.backends.mps.is_available():
    print("MPS accelerator (Metal) found. The Mac GPU will be used.")
elif torch.cuda.is_available():
    print("CUDA GPU found.")
else:
    print("No GPU found, the CPU will be used.")


## 1. Hyperparameter Configuration
Configure TGN hyperparameters for local evaluation (1000 users, 2000 devices, 50,000 events).


In [ ]:
from graphagate.config import TGNConfig
from graphagate.train_tgn import train_tgn

cfg = TGNConfig(
    num_users=1000,
    num_devices=2000,
    num_sources=1500,
    num_configs=400,
    num_events=50000,
    epochs=20,
    batch_size=256,
    eval_batch_size=128,
    capacity_headroom=2000,
)

## 2. Training and Evaluation
Execute the training and evaluation pipeline on the streaming access dataset.


In [ ]:
# Il device (CUDA / MPS / CPU) e la compatibilita' MPS sono gestiti da
# graphagate.mps_compat, richiamato da train_tgn. Non serve piu' alcuna patch qui:
# la precedente conversione a float32 perdeva precisione sui timestamp interi
# (float32 e' esatto solo fino a 2**24; un epoch unix ~1.76e9 sbagliava di ~20 s).
# Per forzare un device:  os.environ['GRAPHAGATE_DEVICE'] = 'cpu' | 'mps' | 'cuda'

# Avvia il training. Gli artefatti vengono salvati in public/
metrics = train_tgn(cfg)

print("\n--- Training Completed ---")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")
